# HOG size and sex bias

Load packages

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go

Add function for Wilson confidence interval, instead of traditional CI, as the proportions for some sizes are 0 or 1. 

In [2]:
def wilson_ci(k, n, z=1.96):
    p = k / n
    denom = 1 + z**2 / n
    center = (p + z**2 / (2*n)) / denom
    margin = z * np.sqrt((p*(1-p) + z**2/(4*n)) / n) / denom
    return center - margin, center + margin

#z = z-score corresponding here to 1.96 for 95% interval.  
#p = p(hat) = proportion of success. 
#k = sucesses 
#n = number of trials 

Load the annotated result dataset from Salmon map

In [3]:
salmon_map_full_annot = pd.read_csv("C:/Users/Sebas/OneDrive/Dokument/Master courses/MASTER THESIS/R project-Master Thesis/salmon_map_dominance_DE_sex_results_new_filtering_genotype_controlled_age_rank_fixed.csv", float_precision='legacy')
# load the full annotation and the lowly expressed datasets for later defining hog size columns 
full_annotation = pd.read_csv("C:/Users/Sebas/OneDrive/Dokument/Master courses/MASTER THESIS/R project-Master Thesis/C_mac_full_annotation_with_age_fixed.csv")
prefilter_df    = pd.read_csv("C:/Users/Sebas/OneDrive/Dokument/Master courses/MASTER THESIS/R project-Master Thesis/prefilter_transcripts_annotated_May27.csv")

salmon_map_full_annot

,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj,transcript_id,gene_id,seqname,chr_loc,...,COG_category,eggNOG_OGs,species_tree_birth_node,gene_tree_birth_node,birth_type,age_rank,node_depth_from_root,branch_length,n_copies_in_og,n_species_in_og
0,248.420188,0.640709,0.110526,5.796904,6.755020e-09,1.316673e-08,g2.t1,g2,utg000001l,A,...,I,"2FBM0@1|root,2TCUK@2759|Eukaryota,398E5@33154|...",C_maculatus_filtered_proteinfasta_TE_filtered,n3,duplication,9.0,0.758700,0.061748,2.0,8.0
1,92.726112,0.650988,0.104454,6.232321,4.595741e-10,9.353175e-10,g3.t1,g3,utg000001l,A,...,DUZ,"KOG2101@1|root,KOG2101@2759|Eukaryota,396Y9@33...",C_maculatus_filtered_proteinfasta_TE_filtered,n9,duplication,9.0,0.758700,0.061748,2.0,14.0
2,136.044931,-0.600974,0.070022,-8.582668,9.269779e-18,2.387530e-17,g4.t1,g4,utg000001l,A,...,H,"COG0181@1|root,KOG2892@2759|Eukaryota,38D6W@33...",C_maculatus_filtered_proteinfasta_TE_filtered,n13,duplication,9.0,0.758700,0.061748,2.0,14.0
3,3.332468,2.699629,0.718636,3.756600,1.722376e-04,2.671748e-04,g5.t1,g5,utg000001l,A,...,I,"KOG2761@1|root,KOG2761@2759|Eukaryota,38F7R@33...",C_maculatus_filtered_proteinfasta_TE_filtered,n8,duplication,9.0,0.758700,0.061748,2.0,12.0
4,381.084979,-0.875080,0.046175,-18.951587,4.284679e-80,3.659726e-79,g6.t1,g6,utg000001l,A,...,B,"2E6V3@1|root,2SDHR@2759|Eukaryota",N8,NaN,mrca_inferred,6.0,0.629385,0.171468,1.0,4.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17569,20.275098,0.319252,0.150056,2.127555,3.337403e-02,4.238130e-02,g34843.t1,g34843,utg003648l,U,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
17570,1.364118,2.068769,0.590743,3.501981,4.618132e-04,6.956768e-04,g34884.t1,g34884,utg003700l,U,...,S,"2D3G7@1|root,2SRFN@2759|Eukaryota,3AMW9@33154|...",C_maculatus_filtered_proteinfasta_TE_filtered,n17,duplication,9.0,0.758700,0.061748,23.0,4.0
17571,8.286716,1.984267,0.355927,5.574921,2.476423e-08,4.718367e-08,g34922.t1,g34922,utg003714l,U,...,NaN,NaN,C_maculatus_filtered_proteinfasta_TE_filtered,n43,duplication,9.0,0.758700,0.061748,22.0,6.0
17572,19.121063,2.284212,0.326813,6.989347,2.761695e-12,6.049015e-12,g35167.t1,g35167,utg003885l,U,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Filtering

In [4]:
print("=== Birth type distribution before filtering ===")
for name, df in [("salmon_map_full_annot", salmon_map_full_annot),
                 ("full_annotation",       full_annotation),
                 ("prefilter_df",          prefilter_df)]:
    print(f"\n{name} (n={len(df):,}):")
    print(df["birth_type"].value_counts(dropna=False).to_string())

# Restrict to confirmed duplication events only.
salmon_map_full_annot = salmon_map_full_annot[salmon_map_full_annot["birth_type"] == "duplication"].copy()
full_annotation       = full_annotation[full_annotation["birth_type"] == "duplication"].copy()
prefilter_df          = prefilter_df[prefilter_df["birth_type"] == "duplication"].copy()

# Add significance and label columns before creating the working dataframe.
salmon_map_full_annot["padj_safe"]     = salmon_map_full_annot["padj"].replace(0, 1e-308).fillna(1)
salmon_map_full_annot["neglog10_padj"] = -np.log10(salmon_map_full_annot["padj_safe"])
salmon_map_full_annot["significant"]   = (
    (salmon_map_full_annot["padj"] < 0.05) &
    (salmon_map_full_annot["log2FoldChange"].abs() > 1)
)
salmon_map_full_annot["label"] = salmon_map_full_annot["gene_id"].where(
    salmon_map_full_annot["significant"], ""
)

# Remove the transcripts without a HOG assignment.
# salmon_map_results_HOG is the main working dataframe for all downstream analyses.
salmon_map_results_HOG = salmon_map_full_annot.dropna(subset=["HOG"]).copy()

print("\n=== After filtering ===")
for name, df in [("salmon_map_full_annot",  salmon_map_full_annot),
                 ("salmon_map_results_HOG",  salmon_map_results_HOG),
                 ("full_annotation",         full_annotation),
                 ("prefilter_df",            prefilter_df)]:
    print(f"{name}: {len(df):,} transcripts")

=== Birth type distribution before filtering ===

salmon_map_full_annot (n=17,574):
birth_type
duplication         8998
mrca_inferred       6028
NaN                 2491
species_specific      57

full_annotation (n=37,988):
birth_type
duplication         20026
NaN                 10670
mrca_inferred        6733
species_specific      559

prefilter_df (n=36,382):
birth_type
duplication         19063
NaN                 10075
mrca_inferred        6733
species_specific      511

=== After filtering ===
salmon_map_full_annot: 8,998 transcripts
salmon_map_results_HOG: 8,989 transcripts
full_annotation: 20,026 transcripts
prefilter_df: 19,063 transcripts


In [5]:
# Genome and mapped sizes use .t1-filtered full_annotation and prefilter_df.
# Expressed size uses salmon_map_results_HOG itself.
hog_size_genome_df    = (full_annotation.dropna(subset=["HOG"])
                         .groupby("HOG").size()
                         .reset_index(name="hog_size_genome"))

hog_size_mapped_df    = (prefilter_df.dropna(subset=["HOG"])
                         .groupby("HOG").size()
                         .reset_index(name="hog_size_mapped"))

hog_size_expressed_df = (salmon_map_results_HOG
                         .groupby("HOG").size()
                         .reset_index(name="hog_size_expressed"))

salmon_map_results_HOG = (salmon_map_results_HOG
                          .merge(hog_size_genome_df,    on="HOG", how="left")
                          .merge(hog_size_mapped_df,    on="HOG", how="left")
                          .merge(hog_size_expressed_df, on="HOG", how="left"))

print("=== HOG size summary (duplication only) ===")
for name, df_sizes in [("genome",    hog_size_genome_df.set_index("HOG")["hog_size_genome"]),
                       ("mapped",    hog_size_mapped_df.set_index("HOG")["hog_size_mapped"]),
                       ("expressed", hog_size_expressed_df.set_index("HOG")["hog_size_expressed"])]:
    print(f"\n{name}:")
    print(f"  total families:     {len(df_sizes):,}")
    print(f"  families >= 2:      {(df_sizes >= 2).sum():,}")
    print(f"  total transcripts:  {df_sizes.sum():,}")
    print(f"  transcripts >= 2:   {df_sizes[df_sizes >= 2].sum():,}")
    print(f"  max family size:    {df_sizes.max()}")
    print(f"  median family size: {df_sizes.median()}")

print("\n=== Sanity check ===")
for name, df in [("full_annotation",       full_annotation),
                 ("prefilter_df",          prefilter_df),
                 ("salmon_map_full_annot", salmon_map_full_annot)]:
    print(f"{name}: {df['HOG'].notna().sum():,} with HOG / {len(df):,} total")

=== HOG size summary (duplication only) ===

genome:
  total families:     5,567
  families >= 2:      4,205
  total transcripts:  20,004
  transcripts >= 2:   18,642
  max family size:    92
  median family size: 2.0

mapped:
  total families:     5,567
  families >= 2:      4,101
  total transcripts:  19,042
  transcripts >= 2:   17,576
  max family size:    78
  median family size: 2.0

expressed:
  total families:     4,359
  families >= 2:      2,554
  total transcripts:  8,989
  transcripts >= 2:   7,184
  max family size:    28
  median family size: 2.0

=== Sanity check ===
full_annotation: 20,004 with HOG / 20,026 total
prefilter_df: 19,042 with HOG / 19,063 total
salmon_map_full_annot: 8,989 with HOG / 8,998 total


Compute number of paralogs in each HOG.  
Add the hog_size columns to the results table by merging on HOG name from the two other datasets. 

In the genome (and mapped) there are 14 402 gene families. The represented in the expressed dataset is 10 807. The remaining are not visible as they are not expressed

Add sex bias column

In [6]:
salmon_map_results_HOG["sex_bias"] = "not_significant"

salmon_map_results_HOG.loc[
    salmon_map_results_HOG["significant"] &
    (salmon_map_results_HOG["log2FoldChange"] > 0),
    "sex_bias"
] = "male"

salmon_map_results_HOG.loc[
    salmon_map_results_HOG["significant"] &
    (salmon_map_results_HOG["log2FoldChange"] < 0),
    "sex_bias"
] = "female"
salmon_map_results_HOG

,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj,transcript_id,gene_id,seqname,chr_loc,...,n_copies_in_og,n_species_in_og,padj_safe,neglog10_padj,significant,label,hog_size_genome,hog_size_mapped,hog_size_expressed,sex_bias
0,248.420188,0.640709,0.110526,5.796904,6.755020e-09,1.316673e-08,g2.t1,g2,utg000001l,A,...,2.0,8.0,1.316673e-08,7.880522,False,,2,2,2,not_significant
1,92.726112,0.650988,0.104454,6.232321,4.595741e-10,9.353175e-10,g3.t1,g3,utg000001l,A,...,2.0,14.0,9.353175e-10,9.029041,False,,2,2,2,not_significant
2,136.044931,-0.600974,0.070022,-8.582668,9.269779e-18,2.387530e-17,g4.t1,g4,utg000001l,A,...,2.0,14.0,2.387530e-17,16.622051,False,,2,2,2,not_significant
3,3.332468,2.699629,0.718636,3.756600,1.722376e-04,2.671748e-04,g5.t1,g5,utg000001l,A,...,2.0,12.0,2.671748e-04,3.573204,True,g5,2,2,2,male
4,97.175715,1.341896,0.124766,10.755269,5.597042e-27,1.811622e-26,g9.t1,g9,utg000001l,A,...,2.0,14.0,1.811622e-26,25.741932,True,g9,2,2,2,male
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8984,309.335990,-0.687209,0.159236,-4.315651,1.591330e-05,2.640215e-05,g34610.t1,g34610,utg003498l,U,...,3.0,14.0,2.640215e-05,4.578361,False,,3,3,3,not_significant
8985,10.439363,0.527491,0.759882,0.694175,4.875723e-01,5.260053e-01,g34647.t1,g34647,utg003542l,U,...,23.0,3.0,5.260053e-01,0.279010,False,,23,21,8,not_significant
8986,1.364118,2.068769,0.590743,3.501981,4.618132e-04,6.956768e-04,g34884.t1,g34884,utg003700l,U,...,23.0,4.0,6.956768e-04,3.157592,True,g34884,23,23,9,male
8987,8.286716,1.984267,0.355927,5.574921,2.476423e-08,4.718367e-08,g34922.t1,g34922,utg003714l,U,...,22.0,6.0,4.718367e-08,7.326208,True,g34922,22,21,8,male


In [7]:
def hog_size_table_from_df(size_df, size_col):
    t = (
        size_df.groupby(size_col)
        .size()
        .reset_index(name="n_HOGs")
        .rename(columns={size_col: "HOG_size"})
    )
    t["n_transcripts"] = t["HOG_size"] * t["n_HOGs"]
    return t.set_index("HOG_size")

genome    = hog_size_table_from_df(hog_size_genome_df,    "hog_size_genome")
mapped    = hog_size_table_from_df(hog_size_mapped_df,    "hog_size_mapped")
expressed = hog_size_table_from_df(hog_size_expressed_df, "hog_size_expressed")

hog_size_comparison = pd.concat(
    [genome, mapped, expressed],
    axis=1,
    keys=["genome", "mapped", "expressed"]
).fillna(0).astype(int)

hog_size_comparison.columns = [
    "Genome gene families",   "Genome transcripts",
    "Mapped gene families",        "Mapped transcripts",
    "Expressed gene families",     "Expressed transcripts",
]

print(sum(hog_size_comparison["Genome gene families"]))
print(sum(hog_size_comparison["Genome transcripts"]))
print(sum(hog_size_comparison["Mapped gene families"]))
print(sum(hog_size_comparison["Mapped transcripts"]))
print(sum(hog_size_comparison["Expressed gene families"]))
print(sum(hog_size_comparison["Expressed transcripts"]))

hog_size_comparison


5567
20004
5567
19042
4359
8989


,Genome gene families,Genome transcripts,Mapped gene families,Mapped transcripts,Expressed gene families,Expressed transcripts
HOG_size,,,,,,
1,1362,1362,1466,1466,1805,1805
2,2110,4220,2091,4182,1772,3544
3,726,2178,708,2124,369,1107
4,388,1552,376,1504,168,672
5,224,1120,216,1080,81,405
6,142,852,135,810,49,294
7,101,707,104,728,30,210
8,98,784,91,728,28,224
9,66,594,57,513,14,126


# Analysis 1: Bias direction among biased transcripts. 
Which fraction is male within each HOG size? 

Which transcripts that belong to a HOG is biased? Remove the unbiased transcripts

In [8]:
biased = salmon_map_results_HOG[
    salmon_map_results_HOG["sex_bias"].isin(["male", "female"])
].copy()

biased
# 4001 belong to a HOG and are significantly sex biased

,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj,transcript_id,gene_id,seqname,chr_loc,...,n_copies_in_og,n_species_in_og,padj_safe,neglog10_padj,significant,label,hog_size_genome,hog_size_mapped,hog_size_expressed,sex_bias
3,3.332468,2.699629,0.718636,3.756600,1.722376e-04,2.671748e-04,g5.t1,g5,utg000001l,A,...,2.0,12.0,2.671748e-04,3.573204,True,g5,2,2,2,male
4,97.175715,1.341896,0.124766,10.755269,5.597042e-27,1.811622e-26,g9.t1,g9,utg000001l,A,...,2.0,14.0,1.811622e-26,25.741932,True,g9,2,2,2,male
7,237.620281,-4.015786,0.186764,-21.501956,1.492615e-102,1.902623e-101,g13.t1,g13,utg000001l,A,...,1.0,14.0,1.902623e-101,100.720647,True,g13,1,1,1,female
9,961.410546,-4.370523,0.152183,-28.718882,2.217240e-181,1.368365e-179,g15.t1,g15,utg000001l,A,...,2.0,14.0,1.368365e-179,178.863798,True,g15,2,2,2,female
14,2.293755,1.483475,0.506652,2.927995,3.411559e-03,4.784316e-03,g34.t1,g34,utg000001l,A,...,3.0,4.0,4.784316e-03,2.320180,True,g34,3,3,2,male
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8980,33.909303,1.648491,0.217331,7.585149,3.321062e-14,7.748702e-14,g34172.t1,g34172,utg003311l,U,...,19.0,1.0,7.748702e-14,13.110771,True,g34172,18,18,4,male
8982,3.429476,2.064888,0.534763,3.861312,1.127796e-04,1.773450e-04,g34258.t1,g34258,utg003361l,U,...,13.0,3.0,1.773450e-04,3.751181,True,g34258,13,13,3,male
8986,1.364118,2.068769,0.590743,3.501981,4.618132e-04,6.956768e-04,g34884.t1,g34884,utg003700l,U,...,23.0,4.0,6.956768e-04,3.157592,True,g34884,23,23,9,male
8987,8.286716,1.984267,0.355927,5.574921,2.476423e-08,4.718367e-08,g34922.t1,g34922,utg003714l,U,...,22.0,6.0,4.718367e-08,7.326208,True,g34922,22,21,8,male


Count male and female biased transcripts per HOG size.  
Filter out HOG size = 1 as those have no paralogs

In [9]:
'''
SWITCH BACK TO hog_size_expressed to get the old plot back!
'''

hog_counts = (
    biased
    .groupby(["hog_size_genome", "sex_bias"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

hog_counts

sex_bias,hog_size_genome,female,male
0,1,174,298
1,2,472,897
2,3,131,375
3,4,57,197
4,5,34,160
5,6,28,142
6,7,43,69
7,8,21,100
8,9,14,60
9,10,12,67


In [10]:
'''
SWITCH BACK TO hog_size_expressed to get the old plot back!
'''
hog_counts = hog_counts[hog_counts["hog_size_genome"] >= 2].copy()
hog_counts

sex_bias,hog_size_genome,female,male
1,2,472,897
2,3,131,375
3,4,57,197
4,5,34,160
5,6,28,142
6,7,43,69
7,8,21,100
8,9,14,60
9,10,12,67
10,11,8,62


Im doing this as a binomial proportion of p_male = male/(male+female) as some sizes have 0 female biased transcripts.  
Confidence intervals calulated with Wilson score interval, performs better when proportions are close to 0 or 1. 

In [11]:
hog_counts["total_biased"] = hog_counts["male"] + hog_counts["female"]

hog_counts["p_male"] = (
    hog_counts["male"] /
    hog_counts["total_biased"]
)

ci_bounds = hog_counts.apply(
    lambda row: wilson_ci(row["male"], row["total_biased"]),
    axis=1
)

hog_counts["ci_lower"] = [c[0] for c in ci_bounds]
hog_counts["ci_upper"] = [c[1] for c in ci_bounds]

#clipping to avoid floating point residues 
hog_counts["ci_lower"] = hog_counts["ci_lower"].clip(lower=0)
hog_counts["ci_upper"] = hog_counts["ci_upper"].clip(upper=1)


#old normal confidence interval
#hog_counts["se"] = np.sqrt(
#    hog_counts["p_male"] *
#    (1 - hog_counts["p_male"]) /
#    hog_counts["total_biased"]
#)

#hog_counts["ci_lower"] = hog_counts["p_male"] - 1.96 * hog_counts["se"]
#hog_counts["ci_upper"] = hog_counts["p_male"] + 1.96 * hog_counts["se"]
hog_counts


sex_bias,hog_size_genome,female,male,total_biased,p_male,ci_lower,ci_upper
1,2,472,897,1369,0.655223,0.629642,0.679935
2,3,131,375,506,0.741107,0.701224,0.777356
3,4,57,197,254,0.775591,0.720396,0.822573
4,5,34,160,194,0.824742,0.765085,0.871788
5,6,28,142,170,0.835294,0.772251,0.883519
6,7,43,69,112,0.616071,0.523573,0.700871
7,8,21,100,121,0.826446,0.749210,0.883592
8,9,14,60,74,0.810811,0.707122,0.883821
9,10,12,67,79,0.848101,0.752999,0.910918
10,11,8,62,70,0.885714,0.790356,0.940939


Plot

In [12]:
'''
SWITCH BACK TO hog_size_expressed to get the old plot back!
'''

fig1 = px.line(
    hog_counts,
    x="hog_size_genome",
    y="p_male",
    markers=True,
    hover_data={
        "male": True,
        "female": True,
        "total_biased": True,
        "p_male": ":.3f"
    }
)

fig1.update_traces(
    error_y=dict(
        type="data",
        symmetric=False,
        array=hog_counts["ci_upper"] - hog_counts["p_male"],
        arrayminus=hog_counts["p_male"] - hog_counts["ci_lower"]
    )
)

fig1.add_hline(
    y=0.5,
    line_dash="dash",
    line_color="black"
)

fig1.update_layout(
    title=dict(text="Proportion of male biased transcripts in each (genom-wide) gene family size", x=0.5, xanchor="center"),
    xaxis_title="Gene family size (≥2)",
    yaxis_title="Proportion (p_male = #male/#male+#female)",
)

fig1.show()

# Analysis 2: Male/Female/Unbiased prop within HOG size.  

In [13]:
'''
SWITCH BACK TO hog_size_expressed to get the old plot back!
'''

#number in each category
hog_bias = (
    salmon_map_results_HOG
    .groupby(["hog_size_genome", "sex_bias"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

#add the total
hog_bias["total"] = (
    hog_bias["male"] +
    hog_bias["female"] +
    hog_bias["not_significant"]
)


hog_bias

sex_bias,hog_size_genome,female,male,not_significant,total
0,1,174,298,790,1262
1,2,472,897,1986,3355
2,3,131,375,562,1068
3,4,57,197,338,592
4,5,34,160,149,343
5,6,28,142,129,299
6,7,43,69,84,196
7,8,21,100,99,220
8,9,14,60,74,148
9,10,12,67,58,137


Filter out HOG size 1

In [14]:
'''
SWITCH BACK TO hog_size_expressed to get the old plot back!
'''

hog_bias = hog_bias[hog_bias["hog_size_genome"] >= 2].copy()
hog_bias

sex_bias,hog_size_genome,female,male,not_significant,total
1,2,472,897,1986,3355
2,3,131,375,562,1068
3,4,57,197,338,592
4,5,34,160,149,343
5,6,28,142,129,299
6,7,43,69,84,196
7,8,21,100,99,220
8,9,14,60,74,148
9,10,12,67,58,137
10,11,8,62,54,124


Convert to long format instead of wide format

In [15]:
'''
SWITCH BACK TO hog_size_expressed to get the old plot back!
'''

hog_bias_long = hog_bias.melt(
    id_vars=["hog_size_genome", "total"],
    value_vars=["male", "female", "not_significant"],
    var_name="sex_bias",
    value_name="transcript_count"
)
hog_bias_long

,hog_size_genome,total,sex_bias,transcript_count
0,2,3355,male,897
1,3,1068,male,375
2,4,592,male,197
3,5,343,male,160
4,6,299,male,142
...,...,...,...,...
151,58,7,not_significant,5
152,60,28,not_significant,16
153,62,23,not_significant,13
154,80,23,not_significant,14


Add the proprtions and confidence intervals.  
Confidence intervals calulated with Wilson score interval, performs better when proportions are close to 0 or 1. 

In [16]:
hog_bias_long["proportion"] = (
    hog_bias_long["transcript_count"] /
    hog_bias_long["total"]
)

ci_bounds = hog_bias_long.apply(
    lambda row: wilson_ci(row["transcript_count"], row["total"]),
    axis=1
)

hog_bias_long["ci_lower"] = [c[0] for c in ci_bounds]
hog_bias_long["ci_upper"] = [c[1] for c in ci_bounds]

#clipping to avoid floating point residues 
hog_bias_long["ci_lower"] = hog_bias_long["ci_lower"].clip(lower=0)
hog_bias_long["ci_upper"] = hog_bias_long["ci_upper"].clip(upper=1)


#old normal confidence interval
#hog_bias_long["se"] = np.sqrt(
#    hog_bias_long["proportion"] *
#    (1 - hog_bias_long["proportion"]) /
#    hog_bias_long["total"]
#)

#hog_bias_long["ci_lower"] = hog_bias_long["proportion"] - 1.96 * hog_bias_long["se"]
#hog_bias_long["ci_upper"] = hog_bias_long["proportion"] + 1.96 * hog_bias_long["se"]
hog_bias_long

,hog_size_genome,total,sex_bias,transcript_count,proportion,ci_lower,ci_upper
0,2,3355,male,897,0.267362,0.252658,0.282598
1,3,1068,male,375,0.351124,0.323076,0.380238
2,4,592,male,197,0.332770,0.295998,0.371699
3,5,343,male,160,0.466472,0.414340,0.519348
4,6,299,male,142,0.474916,0.418990,0.531479
...,...,...,...,...,...,...,...
151,58,7,not_significant,5,0.714286,0.358929,0.917783
152,60,28,not_significant,16,0.571429,0.390705,0.734917
153,62,23,not_significant,13,0.565217,0.368110,0.743656
154,80,23,not_significant,14,0.608696,0.407852,0.778426


Plot with plotly

In [17]:
'''
SWITCH BACK TO hog_size_expressed to get the old plot back!
'''
fig2 = px.line(
    hog_bias_long,
    x="hog_size_genome",
    y="proportion",
    color="sex_bias",
    markers=True,
        hover_data={
        "transcript_count": True,
        "total": True,
        "proportion": ":.3f"
    }
)

for trace in fig2.data:
    bias_type = trace.name
    
    subset = hog_bias_long[hog_bias_long["sex_bias"] == bias_type]
    
    trace.error_y = dict(
        type="data",
        symmetric=False,
        array=subset["ci_upper"].values - subset["proportion"].values,
        arrayminus=subset["proportion"].values - subset["ci_lower"].values
    )

fig2.update_layout(
    title=dict(text="Proportion of transcript bias in each gene family size (genome-wide)", x=0.5, xanchor="center"),
    xaxis_title="Gene family size (≥2)",
    yaxis_title="Proportion of transcripts",
)
fig2.show()

# Analysis 3: Variance vs HOG Size
How variable are the log2FoldChange values inside each HOG size? 

Transcript level log2FC variance per HOG size

In [18]:
#About the expression patterns 
variance_by_size = (
    salmon_map_results_HOG
    .groupby("hog_size_genome")["log2FoldChange"]
    .agg(
        variance="var",
        n="count"
    )
    .reset_index()
)

#filter out size 1
variance_by_size = variance_by_size[
    variance_by_size["hog_size_genome"] >= 2
].copy()

#filter out sizes with 0 transcripts
variance_by_size = variance_by_size[
    variance_by_size["n"] > 0
]


variance_by_size

,hog_size_genome,variance,n
1,2,6.928334,3355
2,3,8.481561,1068
3,4,6.591679,592
4,5,7.802240,343
5,6,8.203123,299
6,7,4.033932,196
7,8,6.414174,220
8,9,5.018230,148
9,10,12.944266,137
10,11,4.704432,124


Plot

In [19]:
fig3 = px.line(
    variance_by_size,
    x="hog_size_genome",
    y="variance",
    markers=True,
    hover_data={
        "n": True,
        "variance": ":.3f"
    }
)

fig3.update_layout(
     title=dict(text="Transcript variance within each (expressed) gene family size", x=0.5, xanchor="center"),
    xaxis_title="(Genome) Gene family size (≥2)",
    yaxis_title="Variance of log2FoldChange",
)
fig3.write_image("transcripts_variance_size.svg")
fig3.show()


Expression differene decreases as the family size increases. But for HOG sizes larger than 15 we have very small sample sizes so variance is noisy and unreliable as small sample makes variance unstable

# Analysis 4: Variance within each HOG

In [20]:
variance_within_hog = (
    salmon_map_results_HOG
    .groupby(["HOG", "hog_size_genome"])["log2FoldChange"]
    .agg(
        variance="var"
    )
    .reset_index()
)

variance_within_hog = variance_within_hog[
    variance_within_hog["hog_size_genome"] >= 2
]

variance_within_hog

,HOG,hog_size_genome,variance
0,N0.HOG0000009,27,1.635579
1,N0.HOG0000011,33,0.324677
2,N0.HOG0000014,56,1.452074
3,N0.HOG0000015,2,0.003742
5,N0.HOG0000017,3,NaN
...,...,...,...
4354,N0.HOG0017481,3,NaN
4355,N0.HOG0017482,3,NaN
4356,N0.HOG0017490,3,0.157018
4357,N0.HOG0017491,3,0.088576


Plot

In [21]:
#separate the outlier 
low = variance_within_hog[variance_within_hog["variance"] <= 120]
high = variance_within_hog[variance_within_hog["variance"] > 120]

#create two subplots 
fig4 = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    row_heights=[0.1, 0.9],  # small top, big bottom
    vertical_spacing=0.05
)

#Add bottom panel (main plot)
fig4.add_trace(
    go.Scatter(
        x=low["hog_size_genome"],
        y=low["variance"],
        mode="markers",
        hovertext=low["HOG"],
        name="Variance"
    ),
    row=2, col=1
)

#Add top panel (outlier)
fig4.add_trace(
    go.Scatter(
        x=high["hog_size_genome"],
        y=high["variance"],
        mode="markers",
        hovertext=high["HOG"],
        name="Outlier"
    ),
    row=1, col=1
)

#set axis rate
fig4.update_yaxes(range=[0,120], row=2, col=1)
fig4.update_yaxes(range=[320,340], 
                  tickmode="array",
                  tickvals=[320,340],
                  row=1, col=1)

fig4.update_layout(
    title=dict(text="Within-family variance across gene family sizes", x=0.5, xanchor="center"),
    height=600,
    width=1200,
    showlegend=False,
)
# Set bottom axis titles
fig4.update_xaxes(range=[0,90], title_text=" Gene family size (genome-wide, size ≥ 2)", row=2, col=1, tickmode="linear", dtick=5)
fig4.update_yaxes(title_text="Variance (log2FoldChange)", row=2, col=1)

#top x axis ticks
fig4.update_xaxes(range=[0,90], row=1, col=1, tickmode="linear", dtick=5)
fig4.write_image("within_family_variance_size.svg")
fig4.show()

In [22]:
hog_members = salmon_map_results_HOG[
    salmon_map_results_HOG["HOG"] == "N0.HOG0000646"
][["transcript_id", "gene_id", "HOG", "OG", "log2FoldChange", 
   "padj", "Description", "PFAMs", "hog_size_expressed", "hog_size_genome"]]

print(f"HOG members: {len(hog_members)}")
print(hog_members.to_string())

HOG members: 3
     transcript_id gene_id            HOG         OG  log2FoldChange          padj                                      Description                        PFAMs  hog_size_expressed  hog_size_genome
4522     g14699.t1  g14699  N0.HOG0000646  OG0000445       -2.350378  1.509481e-07  RNA polymerase II regulatory region DNA binding  DUF659,Dimer_Tnp_hAT,zf-BED                   3               10
5120     g16714.t1  g16714  N0.HOG0000646  OG0000445       29.998306  5.525148e-24  RNA polymerase II regulatory region DNA binding  DUF659,Dimer_Tnp_hAT,zf-BED                   3               10
6004     g19428.t1  g19428  N0.HOG0000646  OG0000445        0.032147  9.729501e-01  RNA polymerase II regulatory region DNA binding  DUF659,Dimer_Tnp_hAT,zf-BED                   3               10


# Analysis 5: Within HOG directional bias 

Put the HOGs into categories (biased_sets)

In [23]:
# this we still want as expressed
hog_direction = (
    salmon_map_results_HOG
    .groupby(["HOG", "hog_size_expressed"])["sex_bias"]
    .apply(lambda x: set(x))
    .reset_index()
)

hog_direction.rename(columns={"sex_bias": "bias_set"}, inplace=True)

#classify the sets
def classify_bias(bias_set):
    
    if bias_set == {"male"}:
        return "All male biased"
    
    elif bias_set == {"female"}:
        return "All female biased"
    
    elif bias_set == {"not_significant"}:
        return "All unbiased"
    
    elif bias_set == {"male", "female"}:
        return "Male + Female"
    
    elif bias_set == {"male", "not_significant"}:
        return "Male + Unbiased"
    
    elif bias_set == {"female", "not_significant"}:
        return "Female + Unbiased"
    
    elif bias_set == {"male", "female", "not_significant"}:
        return "All three"
    
    else:
        return "Other"
    
hog_direction["category"] = hog_direction["bias_set"].apply(classify_bias)
hog_direction = hog_direction[
    hog_direction["hog_size_expressed"] >= 2
]
hog_direction = hog_direction.merge(
    salmon_map_results_HOG[["HOG", "hog_size_genome", "hog_size_mapped"]].drop_duplicates("HOG"),
    on="HOG", how="left"
)

hog_direction

,HOG,hog_size_expressed,bias_set,category,hog_size_genome,hog_size_mapped
0,N0.HOG0000009,9,"{male, not_significant}",Male + Unbiased,27,26
1,N0.HOG0000011,7,"{male, not_significant}",Male + Unbiased,33,33
2,N0.HOG0000014,17,"{male, not_significant}",Male + Unbiased,56,56
3,N0.HOG0000015,2,{female},All female biased,2,2
4,N0.HOG0000020,5,{female},All female biased,5,5
...,...,...,...,...,...,...
2549,N0.HOG0017467,2,"{not_significant, female}",Female + Unbiased,3,3
2550,N0.HOG0017473,2,{not_significant},All unbiased,3,3
2551,N0.HOG0017490,2,{not_significant},All unbiased,3,3
2552,N0.HOG0017491,3,{female},All female biased,3,3


Count categories

In [24]:
category_order = [
    "All three",
    "All male biased",
    "Male + Unbiased",
    "Male + Female",
    "Female + Unbiased",
    "All female biased",
    "All unbiased"
]

category_colors = {
    "All three":        "#B39DDB",
    "All male biased":  "#1F4BFF",
    "Male + Unbiased":  "#4A90E2",
    "Male + Female":    "#C77DFF",   
    "Female + Unbiased":"#F28B82",
    "All female biased":"#D32F2F",
    "All unbiased":     "#66BB6A",
}

hog_direction["category"] = pd.Categorical(
    hog_direction["category"],
    categories=category_order,
    ordered=True
)

hog_category_counts = (
    hog_direction
    .groupby("category", observed=False)
    .size()
    .reset_index(name="count")
)

hog_category_counts

,category,count
0,All three,53
1,All male biased,511
2,Male + Unbiased,515
3,Male + Female,34
4,Female + Unbiased,175
5,All female biased,200
6,All unbiased,1066


Plot

In [25]:
fig5 = px.bar(
    hog_category_counts,
    x="category",
    y="count",
    color="category",
    color_discrete_map=category_colors,
    category_orders={"category": category_order}
)

fig5.update_layout(
    title=dict(text="Number of Gene families in different transcript bias composition categories", x=0.5, xanchor="center"),
    xaxis_title="Gene family composition category",
    yaxis_title="Number of Gene families",
    xaxis_tickangle=45,
    showlegend=False
)

fig5.show()

Get the proportions of bias in each category bar

In [26]:
# Merge category back onto transcript level
transcript_cats = salmon_map_results_HOG.merge(
    hog_direction[["HOG", "category"]],
    on="HOG",
    how="inner"
)

# Count transcripts per category and sex_bias
transcript_composition = (
    transcript_cats
    .groupby(["category", "sex_bias"], observed=False)
    .size()
    .reset_index(name="count")
)

totals = transcript_composition.groupby("category", observed=False)["count"].sum().reset_index(name="total")
transcript_composition = transcript_composition.merge(totals, on="category")
transcript_composition["pct"] = transcript_composition["count"] / transcript_composition["total"] * 100

BIAS_COLORS_3 = {
    "male":            "#6BAED6",
    "female":          "#E07B8A",
    "not_significant": "#708090",
}

bias_order = ["male", "not_significant", "female"]
bias_labels = {
    "male":            "Male-biased",
    "female":          "Female-biased",
    "not_significant": "Unbiased",
}

# Number of HOGs per category
n_hogs_per_category = (
    hog_direction
    .groupby("category", observed=False)
    .size()
    .reset_index(name="n_hogs")
)

# Use transcript proportions per category to split bars
cat_bias = (
    transcript_cats
    .groupby(["category", "sex_bias"], observed=True)
    .size()
    .reset_index(name="count")
)
cat_totals = cat_bias.groupby("category", observed=True)["count"].sum().reset_index(name="total")
cat_bias = cat_bias.merge(cat_totals, on="category")
cat_bias["pct"] = cat_bias["count"] / cat_bias["total"]

# Multiply proportion by n_hogs so bars reach correct height
cat_bias = cat_bias.merge(n_hogs_per_category, on="category")
cat_bias["hog_count"] = cat_bias["pct"] * cat_bias["n_hogs"]

fig5b = go.Figure()

for bias in bias_order:
    sub = cat_bias[cat_bias["sex_bias"] == bias]

    counts_list, texts, customdata = [], [], []
    for cat in category_order:
        row = sub[sub["category"] == cat]
        if len(row) > 0:
            n = row.iloc[0]["hog_count"]
            total = row.iloc[0]["n_hogs"]
            pct = row.iloc[0]["pct"] * 100
            counts_list.append(n)
            texts.append(f"{pct:.1f}%" if pct >= 5 else "")
            customdata.append([round(n), total, round(pct, 1)])
        else:
            counts_list.append(0)
            texts.append("")
            customdata.append([0, 0, 0])

    fig5b.add_trace(go.Bar(
        x=category_order,
        y=counts_list,
        name=bias_labels[bias],
        marker_color=BIAS_COLORS_3[bias],
        marker_line_color="black",
        marker_line_width=0.8,
        text=texts,
        textposition="inside",
        textfont=dict(size=10, color="white", family="Arial Black"),
        customdata=customdata,
        hovertemplate=(
            f"<b>{bias_labels[bias]}</b><br>"
            "Category: %{x}<br>"
            "Proportion: %{customdata[2]}%<br>"
            "Equivalent gene families: %{customdata[0]}<br>"
            "Total gene families in category: %{customdata[1]}<extra></extra>"
        ),
    ))

# n= HOGs per category — positioned just above each bar
for cat in category_order:
    total = n_hogs_per_category[n_hogs_per_category["category"] == cat]["n_hogs"].iloc[0]
    fig5b.add_annotation(
        x=cat,
        y=total,
        yref="y",
        text=f"(n={total})",
        showarrow=False,
        yshift=8,
        font=dict(size=9, color="black"),
        yanchor="bottom",
    )

fig5b.update_layout(
    plot_bgcolor="white",
    barmode="stack",
    title=dict(
        text=(
            "<b>Gene family bias composition by category</b>"
            "<br><sup>Y-axis = number of gene families · "
            "bars split by transcript bias proportion within each category</sup>"
        ),
        x=0.5, xanchor="center"
    ),
    xaxis=dict(
        title="Gene family composition category",
        tickangle=45,
        showgrid=False,
        categoryorder="array",
        categoryarray=category_order,
    ),
    yaxis=dict(
        title="Number of gene families",
        showgrid=True,
        gridcolor="lightgrey",
        zeroline=False,
        range=[0, n_hogs_per_category["n_hogs"].max() * 1.12],
    ),
    legend=dict(
        title="Sex bias",
        orientation="h",
        yanchor="bottom", y=1.02,
        xanchor="right", x=1,
    ),
    margin=dict(b=100, t=100),
    width=1400,
    height=800,
)
fig5b.write_image("gene_family_bias_categories.svg")
fig5b.show()

Quick check

In [27]:
# Summary: gene families and transcript counts per bias category
all_cats = [
    "All unbiased", "All male biased", "Male + Unbiased",
    "Female + Unbiased", "All female biased", "Male + Female", "All three"
]

print(f"{'Category':<22} {'Families':>10} {'Transcripts':>13} {'Male%':>8} {'Female%':>9} {'Unbiased%':>11}")
print("-" * 75)

for cat in all_cats:
    n_families = hog_category_counts[hog_category_counts["category"] == cat]["count"].values[0]
    sub = transcript_cats[transcript_cats["category"] == cat]
    total = len(sub)
    if total == 0:
        print(f"{cat:<22} {n_families:>10} {total:>13}")
        continue
    male     = (sub["sex_bias"] == "male").sum()
    female   = (sub["sex_bias"] == "female").sum()
    unbiased = (sub["sex_bias"] == "not_significant").sum()
    print(f"{cat:<22} {n_families:>10} {total:>13} "
          f"{100*male/total:>7.1f}% {100*female/total:>8.1f}% {100*unbiased/total:>10.1f}%")

Category                 Families   Transcripts    Male%   Female%   Unbiased%
---------------------------------------------------------------------------
All unbiased                 1066          2429     0.0%      0.0%      100.0%
All male biased               511          1250   100.0%      0.0%        0.0%
Male + Unbiased               515          2021    50.1%      0.0%       49.9%
Female + Unbiased             175           588     0.0%     47.3%       52.7%
All female biased             200           434     0.0%    100.0%        0.0%
Male + Female                  34           106    54.7%     45.3%        0.0%
All three                      53           356    35.1%     24.2%       40.7%


Export categories to R for subsetted mixed model analysis

In [ ]:
# Export HOG category assignments for R
# Keep HOG, category, and size variables for joining to model_data

export_cols = ["HOG", "category", "hog_size_expressed", 
               "hog_size_genome", "hog_size_mapped"]

hog_direction_export = hog_direction[export_cols].copy()

# Confirm what you are exporting
print("Rows:", len(hog_direction_export))
print("Categories:\n", hog_direction_export["category"].value_counts())


# Remeber to move this into the R repository 
hog_direction_export.to_csv("hog_direction_categories.csv", index=False)
print("Saved to hog_direction_categories.csv")

hog_direction_export

Rows: 2554
Categories:
 category
All unbiased         1066
Male + Unbiased       515
All male biased       511
All female biased     200
Female + Unbiased     175
All three              53
Male + Female          34
Name: count, dtype: int64
Saved to hog_direction_categories.csv


,HOG,category,hog_size_expressed,hog_size_genome,hog_size_mapped
0,N0.HOG0000009,Male + Unbiased,9,27,26
1,N0.HOG0000011,Male + Unbiased,7,33,33
2,N0.HOG0000014,Male + Unbiased,17,56,56
3,N0.HOG0000015,All female biased,2,2,2
4,N0.HOG0000020,All female biased,5,5,5
...,...,...,...,...,...
2549,N0.HOG0017467,Female + Unbiased,2,3,3
2550,N0.HOG0017473,All unbiased,2,3,3
2551,N0.HOG0017490,All unbiased,2,3,3
2552,N0.HOG0017491,All female biased,3,3,3


Plotly has an issue with the box plots, so here i use go.Figure() to force plotly  

In [29]:
fig6 = go.Figure()

cat_positions = {cat: i for i, cat in enumerate(category_order)}
np.random.seed(42)

for cat in category_order:
    subset = hog_direction[hog_direction["category"] == cat]
    if subset.empty:
        continue

    color  = category_colors[cat]
    x_pos  = cat_positions[cat]

    # Box (no built-in points — we draw them ourselves below)
    fig6.add_trace(go.Box(
        y=subset["hog_size_genome"],
        x=[x_pos] * len(subset),
        name=cat,
        marker_color=color,
        line_color=color,
        fillcolor=color,
        opacity=0.6,
        boxpoints=False,
        width=0.5,
        showlegend=False
    ))

    # Regular points (size > 2) — circles
    reg = subset[subset["hog_size_genome"] != 2]
    if not reg.empty:
        fig6.add_trace(go.Scatter(
            x=x_pos + np.random.uniform(-0.15, 0.15, len(reg)),
            y=reg["hog_size_genome"],
            mode="markers",
            marker=dict(color=color, symbol="circle", size=5, opacity=0.7,
                        line=dict(width=0.5, color="white")),
            showlegend=False,
            text=reg["HOG"].astype(str),
            hovertemplate="HOG: %{text}<br>Size: %{y}<extra></extra>"
        ))

    # HOG size 2 shape - diamond (All three cannot exist at size 2)
    s2 = subset[subset["hog_size_genome"] == 2]
    if not s2.empty:
        fig6.add_trace(go.Scatter(
            x=x_pos + np.random.uniform(-0.15, 0.15, len(s2)),
            y=s2["hog_size_genome"],
            mode="markers",
            marker=dict(color=color, symbol="diamond", size=8, opacity=0.9,
                        line=dict(width=0.8, color="white")),
            showlegend=False,
            text=s2["HOG"].astype(str),
            hovertemplate="HOG: %{text}<br>Size: 2 ◆<extra></extra>"
        ))

fig6.update_layout(
    title=dict(text=" Gene family sizes in different transcript bias composition categories", x=0.5, xanchor="center"),
    xaxis=dict(
        tickmode="array",
        tickvals=list(cat_positions.values()),
        ticktext=list(cat_positions.keys()),
        tickangle=45,
        title="Gene family composition category"
    ),
    yaxis_title="Gene family size (genome)",
    showlegend=False
)
fig6.write_image("categories_boxplot_size.svg")
fig6.show()

In [30]:

hog_dir_melted = hog_direction.melt(
    id_vars=["HOG", "category"],
    value_vars=["hog_size_genome", "hog_size_mapped", "hog_size_expressed"],
    var_name="size_level",
    value_name="HOG_size"
)

size_level_labels = {
    "hog_size_genome":    "Genome",
    "hog_size_mapped":    "Mapped",
    "hog_size_expressed": "Expressed",
}
hog_dir_melted["size_level"] = hog_dir_melted["size_level"].map(size_level_labels)

fig7 = px.box(
    hog_dir_melted,
    x="category",
    y="HOG_size",
    color="size_level",
    color_discrete_map={
        "Genome":    "#9E9E9E",
        "Mapped":    "#C4A35A",
        "Expressed": "#5BA67A",
    },
    category_orders={
        "category":   category_order,
        "size_level": ["Genome", "Mapped", "Expressed"],
    },
    points="all",
    hover_data=["HOG"],
)

fig7.update_layout(
    title=dict(text="Gene family sizes in different transcript bias composition categories, across size levels", x=0.5, xanchor="center"),
    xaxis_title="Gene family composition category",
    yaxis_title="Gene family size",
    xaxis_tickangle=45,
    legend_title_text="Size level",
)
fig7.show()


% proportion bar chart

In [31]:
hog_size_cat = (
    hog_direction
    .groupby(["hog_size_genome", "category"], observed=False)
    .size()
    .reset_index(name="count")
)

size_totals = (
    hog_size_cat.groupby("hog_size_genome")["count"]
    .sum()
    .reset_index(name="total")
)

hog_size_cat = hog_size_cat.merge(size_totals, on="hog_size_genome")
hog_size_cat["pct"] = hog_size_cat["count"] / hog_size_cat["total"] * 100
hog_size_cat["label"] = hog_size_cat["count"].where(hog_size_cat["pct"] >= 4, other="")

fig10 = px.bar(
    hog_size_cat,
    x="hog_size_genome",
    y="pct",
    color="category",
    color_discrete_map=category_colors,
    category_orders={"category": category_order},
    text="label",
    custom_data=["count", "total", "category"],
    barmode="stack"
)

fig10.update_traces(
    textposition="inside",
    insidetextanchor="middle",
    textfont=dict(color="white", size=10),
    hovertemplate=(
        "<b>%{customdata[2]}</b><br>"
        "Gene family size: %{x}<br>"
        "Proportion: %{y:.1f}%<br>"
        "Count: %{customdata[0]}<br>"
        "Total at this size: %{customdata[1]}"
        "<extra></extra>"
    )
)

# Total n= annotations below each bar
annotations = [
    dict(
        x=row.hog_size_genome, y=101,
        xref="x", yref="y",
        text=f"n={row.total}",
        showarrow=False,
        font=dict(size=9, color="#444"),
        yanchor="bottom",
        textangle=-80
    )
    for row in size_totals.itertuples()
]

fig10.update_layout(
    title=dict(text="Proportion of gene family categories in each size (genome)", x=0.5, xanchor="center"),
    xaxis=dict(tickmode="linear", dtick=5, title="Gene family size (genome, sizes ≥ 2)"),
    yaxis=dict(title="Proportion of gene families (%)", range=[0, 111], ticksuffix="%"),
    legend=dict(
    title="Category",
    orientation="h",
    yanchor="bottom", y=1.02,
    xanchor="right", x=1,
),
    annotations=annotations,
    margin=dict(b=60, t=120),
    width=1200,
    height=600
)
fig10.write_image("proportions_categories_size.svg")
fig10.show()

Density curve plot

Data from prefiltered/unexpressed transcripts. These are post tximport in R, so they are transcrtipts that were still mapped by Salmon. The raw orthofinder data includes even more transcripts, but these have no mapping evidence.

In [32]:
def get_independent_sizes(df, hog_col="HOG"):
    return (
        df.dropna(subset=[hog_col])
        .groupby(hog_col)
        .size()
    )

sizes_genome    = get_independent_sizes(full_annotation)
sizes_mapped    = get_independent_sizes(prefilter_df)
sizes_expressed = get_independent_sizes(salmon_map_results_HOG)

density_datasets = {
    f"Genome (n={( sizes_genome    >= 2).sum():,} Gene families)": (sizes_genome[sizes_genome >= 2],       "#9E9E9E", "rgba(158,158,158,0.2)"),
    f"Mapped (n={(sizes_mapped     >= 2).sum():,} Gene families)": (sizes_mapped[sizes_mapped >= 2],       "#C4A35A", "rgba(196,163,90,0.2)"),
    f"Expressed (n={(sizes_expressed >= 2).sum():,} Gene families)": (sizes_expressed[sizes_expressed >= 2], "#5BA67A", "rgba(91,166,122,0.2)"),
}

fig12 = go.Figure()

for label, (sizes, color, fillcolor) in density_datasets.items():
    counts = sizes.value_counts().sort_index()
    fig12.add_trace(go.Scatter(
        x=counts.index,
        y=counts.values,
        mode="lines+markers",
        name=label,
        fill="tozeroy",
        line=dict(color=color, width=2),
        fillcolor=fillcolor,
        marker=dict(size=4),
    ))

fig12.update_layout(
    title=dict(
        text="Gene family size distribution across dataset levels (size ≥ 2)",
        x=0.5, xanchor="center"
    ),
    xaxis=dict(title="Gene family size", tickmode="linear", dtick=5),
    yaxis=dict(title="Number of gene families"),
    legend_title_text="Dataset level",
    plot_bgcolor="white",
    width=800,
    height=400,
)
fig12.show()

Log scaled

In [33]:
def get_independent_sizes(df, hog_col="HOG"):
    return (
        df.dropna(subset=[hog_col])
        .groupby(hog_col)
        .size()
    )

sizes_genome    = get_independent_sizes(full_annotation)
sizes_mapped    = get_independent_sizes(prefilter_df)
sizes_expressed = get_independent_sizes(salmon_map_results_HOG)

density_datasets = {
    f"Genome (n={( sizes_genome    >= 2).sum():,} Gene families)": (sizes_genome[sizes_genome >= 2],       "#9E9E9E", "rgba(158,158,158,0.2)"),
    f"Mapped (n={(sizes_mapped     >= 2).sum():,} Gene families)": (sizes_mapped[sizes_mapped >= 2],       "#C4A35A", "rgba(196,163,90,0.2)"),
    f"Expressed (n={(sizes_expressed >= 2).sum():,} Gene families)": (sizes_expressed[sizes_expressed >= 2], "#5BA67A", "rgba(91,166,122,0.2)"),
}

fig13 = go.Figure()

for label, (sizes, color, fillcolor) in density_datasets.items():
    counts = sizes.value_counts().sort_index()
    fig13.add_trace(go.Scatter(
        x=counts.index,
        y=counts.values,
        mode="lines+markers",
        name=label,
        fill="tozeroy",
        line=dict(color=color, width=2),
        fillcolor=fillcolor,
        marker=dict(size=4),
    ))

fig13.update_layout(
    title=dict(
        text="Gene family size distribution across dataset levels (size ≥ 2)",
        x=0.5, xanchor="center"
    ),
    xaxis=dict(title="Gene family size", tickmode="linear", dtick=5),
    yaxis=dict(
        title="Number of gene families (log-scaled)",
        type="log",
        tickmode="array",
        tickvals=[1, 10, 100, 1000, 10000],
        ticktext=["1", "10", "100", "1000", "10000"]
        ),
    legend_title_text="Dataset level",
    plot_bgcolor="white",
    width=800,
    height=400,
)

fig13.show()


Subplots for convenience

In [34]:
from plotly.subplots import make_subplots

def get_independent_sizes(df, hog_col="HOG"):
    return (
        df.dropna(subset=[hog_col])
        .groupby(hog_col)
        .size()
    )

sizes_genome    = get_independent_sizes(full_annotation)
sizes_mapped    = get_independent_sizes(prefilter_df)
sizes_expressed = get_independent_sizes(salmon_map_results_HOG)

density_datasets = {
    f"Genome (n={( sizes_genome    >= 2).sum():,} Gene families)": (sizes_genome[sizes_genome >= 2],       "#9E9E9E", "rgba(158,158,158,0.2)"),
    f"Mapped (n={(sizes_mapped     >= 2).sum():,} Gene families)": (sizes_mapped[sizes_mapped >= 2],       "#C4A35A", "rgba(196,163,90,0.2)"),
    f"Expressed (n={(sizes_expressed >= 2).sum():,} Gene families)": (sizes_expressed[sizes_expressed >= 2], "#5BA67A", "rgba(91,166,122,0.2)"),
}

fig = make_subplots(
    rows=1, cols=2,
    shared_xaxes=False,
    subplot_titles=("Linear scale", "Log scale"),
    horizontal_spacing=0.12,
)

for i, (label, (sizes, color, fillcolor)) in enumerate(density_datasets.items()):
    counts = sizes.value_counts().sort_index()

    # Linear subplot
    fig.add_trace(go.Scatter(
        x=counts.index,
        y=counts.values,
        mode="lines+markers",
        name=label,
        fill="tozeroy",
        line=dict(color=color, width=2),
        fillcolor=fillcolor,
        marker=dict(size=4),
        legendgroup=label,
        showlegend=True,
    ), row=1, col=1)

    # Log subplot — reuse same legend group, hide duplicate legend entry
    fig.add_trace(go.Scatter(
        x=counts.index,
        y=counts.values,
        mode="lines+markers",
        name=label,
        fill="tozeroy",
        line=dict(color=color, width=2),
        fillcolor=fillcolor,
        marker=dict(size=4),
        legendgroup=label,
        showlegend=False,
    ), row=1, col=2)

fig.update_xaxes(title_text="Gene family size", tickmode="linear", dtick=5)

fig.update_yaxes(title_text="Number of gene families", row=1, col=1)
fig.update_yaxes(
    title_text="Number of gene families (log-scaled)",
    type="log",
    tickmode="array",
    tickvals=[1, 10, 100, 1000, 10000],
    ticktext=["1", "10", "100", "1,000", "10,000"],
    row=1, col=2,
)

fig.update_layout(
    title=dict(
        text="Gene family size distribution across dataset levels (size ≥ 2)",
        x=0.5, xanchor="center"
    ),
    legend_title_text="Dataset level",
    plot_bgcolor="white",
    width=1400,
    height=450,
)
fig.write_image("size_distribution_subplot.svg")
fig.show()

# Expressed vs unexpressed mapped transcripts per HOG size 

In [35]:
# build the transcript level classification from full_annotation
mapped_ids = set(prefilter_df["transcript_id"])
expressed_ids = set(salmon_map_results_HOG["transcript_id"])

annot = full_annotation.dropna(subset=["HOG"]).copy()
annot["hog_size_genome"] = annot.groupby("HOG")["HOG"].transform("count")

def classify_level(tid):
    if tid in expressed_ids:
        return "Expressed"
    elif tid in mapped_ids:
        return "Mapped, not expressed"
    else: 
        return "Not mapped"

annot["level"] = annot["transcript_id"].apply(classify_level)

level_order = ["Expressed", "Mapped, not expressed", "Not mapped"]
level_colors = {
    "Expressed":              "#5BA67A",
    "Mapped, not expressed":  "#C4A35A",
    "Not mapped":             "#9E9E9E",
}

# Compute proportions per genome gene family size
level_counts = (
    annot[annot["hog_size_genome"] >= 2]
    .groupby(["hog_size_genome", "level"])
    .size()
    .reset_index(name="count")
)

size_totals = level_counts.groupby("hog_size_genome")["count"].sum().reset_index(name="total")
level_counts = level_counts.merge(size_totals, on="hog_size_genome")
level_counts["pct"]   = level_counts["count"] / level_counts["total"] * 100
level_counts["label"] = level_counts["count"].where(level_counts["pct"] >= 4, other="")

fig15 = px.bar(
    level_counts,
    x="hog_size_genome",
    y="pct",
    color="level",
    color_discrete_map=level_colors,
    category_orders={"level": level_order},
    text="label",
    custom_data=["count", "total", "level"],
    barmode="stack"
)

fig15.update_traces(
    textposition="inside",
    insidetextanchor="middle",
    textfont=dict(color="white", size=11),
    hovertemplate=(
        "<b>%{customdata[2]}</b><br>"
        "Gene family size: %{x}<br>"
        "Proportion: %{y:.1f}%<br>"
        "Count: %{customdata[0]}<br>"
        "Total at this size: %{customdata[1]}"
        "<extra></extra>"
    )
)

annotations = [
    dict(
        x=row.hog_size_genome, y=101,
        xref="x", yref="y",
        text=f"<i>n={row.total}</i>",
        showarrow=False,
        font=dict(size=8, color="#444"),
        yanchor="bottom",
        textangle=-80
    )
    for row in size_totals.itertuples()
]

fig15.update_layout(
    title=dict(
        text="Transcript expression level by gene family size (genome)",
        x=0.5, xanchor="center"
    ),
    xaxis=dict(tickmode="linear", dtick=5, title="Gene family size (genome, sizes ≥ 2)"),
    yaxis=dict(title="Proportion of transcripts (%)", range=[0, 110], ticksuffix="%"),
    legend=dict(
        title="Level: ",
        orientation="h",
        yanchor="bottom", y=1.02,
        xanchor="right", x=1,
    ),
    annotations=annotations,
    margin=dict(b=80, t=120),
    width=1300,
    height=600,
    plot_bgcolor="white",
)

# add mean line for the expressed transcripts. This is based on the weighted mean
expressed_rows = level_counts[level_counts["level"] == "Expressed"]
#mean_expressed_pct_weighted = expressed_rows["count"].sum() / expressed_rows["total"].sum() * 100
mean_expressed_pct = expressed_rows["pct"].mean()

fig15.add_hline(
    y=mean_expressed_pct,
    line_color="red",
    line_width=1.5,
    line_dash="dash",
    annotation_text=f"Mean expressed: {mean_expressed_pct:.1f}%",
    annotation_position="top right",
    annotation_font=dict(color="red", size=11),
)
fig15.write_image("proportion_expression_sizes.svg")
fig15.show()


In [36]:
for name, sizes in [("genome", sizes_genome), ("mapped", sizes_mapped), ("expressed", sizes_expressed)]:
    fam = (sizes >= 2).sum()
    trans = sizes[sizes >= 2].sum()
    print(f"{name}: {fam:,} families, {trans:,} transcripts")

genome: 4,205 families, 18,642 transcripts
mapped: 4,101 families, 17,576 transcripts
expressed: 2,554 families, 7,184 transcripts


# Plot each transcript vs. HOG size 

In [37]:
salmon_map_results_HOG_filtered = salmon_map_results_HOG[salmon_map_results_HOG["hog_size_genome"] >= 2].copy()

np.random.seed(42)
jitter = np.random.uniform(-0.15, 0.15, size=len(salmon_map_results_HOG_filtered))

fig17 = px.scatter(
    salmon_map_results_HOG_filtered,
    x=salmon_map_results_HOG_filtered["hog_size_genome"] + jitter,
    y="log2FoldChange",
    opacity=0.4,
    color="sex_bias",
    color_discrete_map={
        "male":            "#6BAED6",
        "female":          "#E07B8A",
        "not_significant": "#9E9E9E",
    },
    hover_data={"transcript_id": True, "HOG": True, "hog_size_expressed": True},
    labels={
        "x":             "Gene family size (genome)",
        "log2FoldChange": "Transcript log2FoldChange",
        "hog_size_expressed": "Gene family size (expressed)"
    },
    title="Transcript-level sex bias as a function of gene family size (genome)"
)

fig17.add_hline(y=0, line_dash="dash", line_color="black")
fig17.update_layout(
    xaxis=dict(tickmode="linear", dtick=5, title="Gene family size (genome, size ≥ 2)"),
    plot_bgcolor="white",
    legend_title_text="Sex bias",
    width=1200
    
)
fig17.write_image("transcript_expression_size.svg")
fig17.show()

In [38]:
salmon_map_results_HOG[salmon_map_results_HOG["HOG"] == "N0.HOG0000646"][
    ["transcript_id", "log2FoldChange", "padj", "hog_size_expressed", "hog_size_genome"]
]

,transcript_id,log2FoldChange,padj,hog_size_expressed,hog_size_genome
4522,g14699.t1,-2.350378,1.509481e-07,3,10
5120,g16714.t1,29.998306,5.525148e-24,3,10
6004,g19428.t1,0.032147,9.729501e-01,3,10
